# Lecture 12

In [5]:
#!/usr/bin/env python3
"""
Tabular LIME demo + weighted LASSO path per instance (cluster centers by class),
with diagnostic scatter panels AND beta-similarity heatmaps.

Dataset: sklearn Breast Cancer (30 features, binary).
Model: sklearn MLPClassifier.
Explanation: LIME-like binary masks + reconstruction by sampling feature marginals.
Importance: Weighted LASSO path on interpretable features (mask indicators).

Outputs:
  - Prints NN test accuracy/AUC
  - For each (class, center): prints top features by path-AUC and diagnostics
  - Saves coefficient-path plots to out_dir/ as PDF
  - Saves 3-panel diagnostic scatter plot to out_dir/ as PDF:
        (1) kappa vs Neff
        (2) curvRMSE vs Neff
        (3) curvRMSE vs kappa
  - Saves beta similarity heatmaps to out_dir/ as PDF:
        (a) cosine similarity between beta vectors
        (b) L2 distance between beta vectors

Notes:
  - "beta" here is taken as the LASSO solution at the *smallest alpha* in the path
    (i.e., least regularized along the returned grid). This is a simple, consistent
    way to compare local surrogates.
"""

import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import lasso_path


# ----------------------------
# Utilities
# ----------------------------
def nearest_real_sample(X: np.ndarray, center: np.ndarray) -> int:
    """Index of sample in X closest to center (Euclidean)."""
    d2 = np.sum((X - center) ** 2, axis=1)
    return int(np.argmin(d2))


def make_marginal_sampler(X_ref: np.ndarray, rng: np.random.Generator):
    """
    Returns sampler(n_samples) that draws vectors by sampling each
    feature independently from empirical marginals of X_ref (already standardized).
    """
    n, d = X_ref.shape

    def sampler(n_samples: int) -> np.ndarray:
        idx = rng.integers(0, n, size=(n_samples, d))
        cols = np.arange(d)[None, :]
        return X_ref[idx, cols]

    return sampler


def effective_sample_size(w: np.ndarray) -> float:
    w = np.asarray(w, dtype=float)
    return float((w.sum() ** 2) / (np.sum(w ** 2) + 1e-12))


def weighted_gram_kappa(Z: np.ndarray, w: np.ndarray, ridge: float = 1e-10) -> float:
    """
    Condition number of G = Z^T W Z (mask space).
    Ridge stabilizes nearly-singular cases.
    """
    w = np.asarray(w, dtype=float)
    Z = np.asarray(Z, dtype=float)
    WZ = Z * w[:, None]
    G = Z.T @ WZ
    G = 0.5 * (G + G.T) + ridge * np.eye(G.shape[0])

    s = np.linalg.svd(G, compute_uv=False)
    smax = max(float(s[0]), 1e-30)
    smin = max(float(s[-1]), 1e-30)
    return float(smax / smin)


def weighted_linear_fit_with_intercept(Z: np.ndarray, y: np.ndarray, w: np.ndarray):
    """
    Fit y ~ b0 + Z b by weighted least squares.
    Returns (b0, b, yhat, rmse_w).
    """
    w = np.clip(np.asarray(w, dtype=float), 1e-12, None)
    Z = np.asarray(Z, dtype=float)
    y = np.asarray(y, dtype=float)

    sw = np.sqrt(w)
    X = np.column_stack([np.ones(Z.shape[0]), Z])

    Xw = X * sw[:, None]
    yw = y * sw

    theta, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
    yhat = X @ theta

    rmse_w = float(np.sqrt(np.sum(w * (y - yhat) ** 2) / np.sum(w)))
    b0 = float(theta[0])
    b = theta[1:].copy()
    return b0, b, yhat, rmse_w


def rank_by_entry(alphas, coefs, feature_names, tol=1e-6):
    """
    Rank features by the alpha at which they first become nonzero.
    Earlier entry (larger alpha) = more important.
    Returns list of (feature_name, entry_alpha, feature_index)
    """
    d, _ = coefs.shape
    entry_alpha = []
    for j in range(d):
        coef_path = coefs[j, :]
        nonzero = np.where(np.abs(coef_path) > tol)[0]
        if len(nonzero) == 0:
            entry_alpha.append((feature_names[j], -np.inf, j))
        else:
            first_idx = nonzero[0]
            entry_alpha.append((feature_names[j], float(alphas[first_idx]), j))
    entry_alpha.sort(key=lambda x: x[1], reverse=True)
    return entry_alpha


# ----------------------------
# LIME perturbations (tabular)
# ----------------------------
def lime_masks_and_reconstructions(
    x: np.ndarray,
    X_ref: np.ndarray,
    n_pert: int = 4000,
    p_keep: float = 0.5,
    sigma: float = 1.0,
    rng: np.random.Generator | None = None,
):
    """
    Interpretable rep: mask Z in {0,1}^d.
    Reconstruction: Xp = Z*x + (1-Z)*X_bg, X_bg sampled from marginals.
    Kernel: w_i = exp(-||Xp_i - x||^2 / sigma^2) in standardized input space.
    """
    if rng is None:
        rng = np.random.default_rng(0)

    d = x.shape[0]
    Z = (rng.random(size=(n_pert, d)) < p_keep).astype(float)

    bg_sampler = make_marginal_sampler(X_ref, rng)
    X_bg = bg_sampler(n_pert)

    Xp = Z * x[None, :] + (1.0 - Z) * X_bg

    dist2 = np.sum((Xp - x[None, :]) ** 2, axis=1)
    w = np.exp(-dist2 / (sigma**2))
    return Z, Xp, w


# ----------------------------
# Weighted LASSO path
# ----------------------------
def weighted_lasso_path(
    Z: np.ndarray,
    y: np.ndarray,
    w: np.ndarray,
    eps: float = 1e-3,
    n_alphas: int = 80,
):
    """
    Weighted LASSO path using sqrt(w) scaling:
      Zw = sqrt(w)*Z, yw = sqrt(w)*y
    Then center Zw,yw to absorb intercept.
    """
    sw = np.sqrt(np.clip(w, 1e-12, None))
    Zw = Z * sw[:, None]
    yw = y * sw

    Zw = Zw - Zw.mean(axis=0, keepdims=True)
    yw = yw - yw.mean()

    alphas, coefs, _ = lasso_path(Zw, yw, eps=eps, n_alphas=n_alphas)
    return alphas, coefs


def summarize_path(
    alphas: np.ndarray,
    coefs: np.ndarray,
    feature_names: list[str],
    top_k: int = 8,
):
    """
    Rank features by area-under-absolute-coefficient path.
    coefs shape: (d, n_alphas)
    """
    abs_coefs = np.abs(coefs)
    auc = np.abs(np.trapezoid(abs_coefs, x=alphas, axis=1))
    order = np.argsort(-auc)
    return [(feature_names[i], float(auc[i]), int(i)) for i in order[:top_k]]


def plot_path(
    alphas,
    coefs,
    feature_names,
    title,
    outpath,
    max_lines=12,
):
    """
    Plot LASSO path + right-side ranking panel by entry order.
    Saves to outpath (PDF).
    """
    ranked = rank_by_entry(alphas, coefs, feature_names)
    top_ranked = ranked[:max_lines]
    top_idx = [idx for _, _, idx in top_ranked]

    fig, axes = plt.subplots(
        1, 2, figsize=(13, 6),
        gridspec_kw={"width_ratios": [2.5, 1]},
    )
    ax_path, ax_text = axes

    for i in top_idx:
        ax_path.plot(alphas, coefs[i, :], label=feature_names[i])

    ax_path.invert_xaxis()
    ax_path.set_xlabel(r"LASSO penalty $\alpha$ (decreasing $\rightarrow$)")
    ax_path.set_ylabel("Coefficient")
    ax_path.set_title(title)
    ax_path.legend(fontsize=8)

    ax_text.axis("off")
    lines = []
    for k, (name, alpha_entry, _) in enumerate(top_ranked, start=1):
        if alpha_entry == -np.inf:
            lines.append(f"{k:2d}. {name}  (never enters)")
        else:
            lines.append(f"{k:2d}. {name}")
    ax_text.text(
        0.0,
        1.0,
        "Entry Order Ranking\n\n" + "\n".join(lines),
        fontsize=10,
        va="top",
        family="monospace",
    )

    plt.tight_layout()
    plt.savefig(outpath)
    plt.close()


# ----------------------------
# Diagnostic scatter panels
# ----------------------------
def plot_diagnostic_scatter_panels(rows, outpath_pdf):
    """
    rows: list of tuples (cls, ci, neff, kappa, curv_rmse)
    Saves 3 panels to a single PDF:
      (1) kappa vs Neff
      (2) curvRMSE vs Neff
      (3) curvRMSE vs kappa
    Class color fixed: 0=blue, 1=orange.
    Center index by marker shape.
    """
    rows = list(rows)
    cls = np.array([r[0] for r in rows], dtype=int)
    ci = np.array([r[1] for r in rows], dtype=int)
    neff = np.array([r[2] for r in rows], dtype=float)
    kappa = np.array([r[3] for r in rows], dtype=float)
    curv = np.array([r[4] for r in rows], dtype=float)

    markers = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]
    class_colors = {0: "blue", 1: "orange"}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    ax1, ax2, ax3 = axes

    def scatter_by_group(ax, x, y, xlabel, ylabel, title, xlog=False, ylog=False):
        for c in [0, 1]:
            for center in np.unique(ci[cls == c]):
                mk = markers[int(center) % len(markers)]
                mask = (cls == c) & (ci == center)
                ax.scatter(
                    x[mask],
                    y[mask],
                    c=class_colors[c],
                    marker=mk,
                    s=75,
                    edgecolors="black",
                    linewidths=0.6,
                    label=f"class {c}, center {int(center)}",
                )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if xlog:
            ax.set_xscale("log")
        if ylog:
            ax.set_yscale("log")

        handles, labels = ax.get_legend_handles_labels()
        uniq = dict(zip(labels, handles))
        ax.legend(uniq.values(), uniq.keys(), fontsize=8, ncol=2, loc="best")

    scatter_by_group(
        ax1,
        x=neff,
        y=kappa,
        xlabel="Neff (effective sample size)",
        ylabel=r"$\kappa(Z^\top W Z)$",
        title=r"$\kappa$ vs Neff",
        xlog=True,
        ylog=True,
    )
    scatter_by_group(
        ax2,
        x=neff,
        y=curv,
        xlabel="Neff (effective sample size)",
        ylabel="curvature_RMSE",
        title="curvRMSE vs Neff",
        xlog=True,
        ylog=False,
    )
    scatter_by_group(
        ax3,
        x=kappa,
        y=curv,
        xlabel=r"$\kappa(Z^\top W Z)$",
        ylabel="curvature_RMSE",
        title="curvRMSE vs kappa",
        xlog=True,
        ylog=False,
    )

    plt.tight_layout()
    plt.savefig(outpath_pdf)
    plt.close()


# ----------------------------
# Beta similarity heatmaps
# ----------------------------
def cosine_similarity_matrix(B: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    B: (m, d) rows are beta vectors.
    Returns (m, m) cosine similarity.
    """
    norms = np.linalg.norm(B, axis=1, keepdims=True)
    norms = np.clip(norms, eps, None)
    U = B / norms
    return U @ U.T


def l2_distance_matrix(B: np.ndarray) -> np.ndarray:
    """
    B: (m, d) rows are beta vectors.
    Returns (m, m) pairwise L2 distances.
    """
    # ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a·b
    s = np.sum(B * B, axis=1, keepdims=True)  # (m,1)
    D2 = s + s.T - 2.0 * (B @ B.T)
    D2 = np.maximum(D2, 0.0)
    return np.sqrt(D2)


def plot_heatmap(M: np.ndarray, labels: list[str], title: str, outpath_pdf: str):
    fig, ax = plt.subplots(figsize=(8.5, 7.5))
    im = ax.imshow(M, aspect="auto")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(outpath_pdf)
    plt.close()


# ----------------------------
# Main
# ----------------------------
def main(
    out_dir: str = "figs/lime_lasso_paths",
    n_centers_per_class: int = 3,
    n_pert: int = 5000,
    p_keep: float = 0.5,
    sigma: float = 1.5,
    random_state: int = 0,
):
    os.makedirs(out_dir, exist_ok=True)
    rng = np.random.default_rng(random_state)

    data = load_breast_cancer()
    X = data.data
    y = data.target
    feature_names = list(data.feature_names)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=random_state, stratify=y
    )

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    clf = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        max_iter=400,
        random_state=random_state,
        early_stopping=True,
        n_iter_no_change=15,
    )
    clf.fit(X_train_s, y_train)

    p_test = clf.predict_proba(X_test_s)[:, 1]
    yhat = (p_test >= 0.5).astype(int)
    acc = accuracy_score(y_test, yhat)
    auc = roc_auc_score(y_test, p_test)
    print(f"Test acc={acc:.3f}  AUC={auc:.3f}")

    # collect diagnostics for scatter panels
    rows = []  # (cls, ci, neff, kappa, curv_rmse)

    # collect betas for similarity heatmaps
    beta_list = []
    beta_labels = []

    # Choose cluster centers per class in standardized space
    for cls in [0, 1]:
        Xc = X_train_s[y_train == cls]
        km = KMeans(n_clusters=n_centers_per_class, random_state=random_state, n_init="auto")
        km.fit(Xc)
        centers = km.cluster_centers_

        for ci, center in enumerate(centers):
            idx_local = nearest_real_sample(Xc, center)
            x = Xc[idx_local]  # standardized instance

            Z, Xp, w = lime_masks_and_reconstructions(
                x=x,
                X_ref=X_train_s,
                n_pert=n_pert,
                p_keep=p_keep,
                sigma=sigma,
                rng=rng,
            )

            # explain model probability of class 1 (your current script’s target)
            y_model = clf.predict_proba(Xp)[:, 1]

            neff = effective_sample_size(w)
            if neff < 50:
                print(f"[warn] class={cls} center={ci}: low Neff={neff:.1f}. Consider increasing sigma.")

            kappa = weighted_gram_kappa(Z, w)

            # curvature proxy in mask space: best weighted linear fit RMSE
            _, _, _, curv_rmse = weighted_linear_fit_with_intercept(Z, y_model, w)

            rows.append((cls, ci, neff, kappa, curv_rmse))

            alphas, coefs = weighted_lasso_path(Z, y_model, w, eps=1e-3, n_alphas=90)

            # --- choose a single beta vector for comparison ---
            beta = coefs[:, -1].copy()  # least regularized end of the returned path
            beta_list.append(beta)
            beta_labels.append(f"c{cls}-k{ci}")

            top = summarize_path(alphas, coefs, feature_names, top_k=10)
            print("\n" + "-" * 70)
            print(f"class={cls}  center={ci}  (nearest training sample in class)")
            print(f"Neff={neff:.1f}   kappa(Z^T W Z)={kappa:.3e}   curvRMSE={curv_rmse:.4f}")
            print("Top features by |coef|-path AUC:")
            for name, score, _ in top:
                print(f"  {name:30s}  auc={score:.4e}")

            fig_path = os.path.join(out_dir, f"path_class{cls}_center{ci}.pdf")
            plot_path(
                alphas,
                coefs,
                feature_names,
                title=f"LIME + weighted LASSO path (class {cls}, center {ci})",
                outpath=fig_path,
                max_lines=12,
            )

    # diagnostic scatter panels (PDF)
    diag_path = os.path.join(out_dir, "diag_scatter_panels.pdf")
    plot_diagnostic_scatter_panels(rows, diag_path)
    print(f"\nSaved diagnostic scatter panels to: {diag_path}")

    # beta similarity heatmaps (PDF)
    B = np.vstack(beta_list)  # (m, d)
    cosM = cosine_similarity_matrix(B)
    l2M = l2_distance_matrix(B)

    cos_path = os.path.join(out_dir, "beta_cosine_similarity.pdf")
    l2_path = os.path.join(out_dir, "beta_l2_distance.pdf")

    plot_heatmap(cosM, beta_labels, "Cosine similarity between local $\\beta$ vectors", cos_path)
    plot_heatmap(l2M, beta_labels, "L2 distance between local $\\beta$ vectors", l2_path)

    print(f"Saved beta cosine similarity heatmap to: {cos_path}")
    print(f"Saved beta L2 distance heatmap to: {l2_path}")

    print(f"\nSaved coefficient-path plots to: {out_dir}/")


if __name__ == "__main__":
    main()

Test acc=0.916  AUC=0.966

----------------------------------------------------------------------
class=0  center=0  (nearest training sample in class)
Neff=202.5   kappa(Z^T W Z)=8.779e+01   curvRMSE=0.0597
Top features by |coef|-path AUC:
  mean fractal dimension          auc=6.1911e-05
  worst fractal dimension         auc=1.4806e-05
  mean symmetry                   auc=1.0715e-05
  worst concave points            auc=7.9903e-06
  mean concave points             auc=7.2868e-06
  radius error                    auc=4.0705e-06
  worst compactness               auc=3.4183e-06
  worst area                      auc=3.2216e-06
  mean smoothness                 auc=3.0918e-06
  worst radius                    auc=2.4483e-06

----------------------------------------------------------------------
class=0  center=1  (nearest training sample in class)
Neff=213.3   kappa(Z^T W Z)=8.498e+01   curvRMSE=0.0681
Top features by |coef|-path AUC:
  mean perimeter                  auc=2.1840e-05
  wor

In [4]:
coefs

NameError: name 'coefs' is not defined